In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

¡Diagnóstico completado! El resultado del HTML nos confirma exactamente lo que sospechábamos: el sitio tiene una capa de optimización (LiteSpeed) y carga dinámica que bloquea las peticiones simples de requests.

Cuando Python leyó la página, solo vio el "esqueleto" (por eso solo encontró el título del "Carrito"), pero el contenido de las cervezas aún no se había "dibujado" en la pantalla. Para Python, la estantería estaba vacía.

La Solución: Subir de nivel a Selenium
Como mencionamos antes, Selenium es la herramienta que usan los profesionales para estos casos. Vamos a engañar al sitio abriendo un navegador real (Chrome) que espere a que las cervezas aparezcan.



In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://cultocervecero.com/categoria-producto/cervezas/cervecerias/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Imprime un pedazo del HTML para ver qué etiquetas hay
print("--- INICIO DEL HTML RECIBIDO ---")
print(response.text[:1000]) 
print("--- FIN DEL HTML ---")

# Intento de encontrar CUALQUIER h2 (donde suelen ir los nombres)
titulos = soup.find_all('h2')
print(f"\nSe encontraron {len(titulos)} etiquetas H2. Aquí las primeras 3:")
for t in titulos[:3]:
    print(f"- {t.text.strip()}")

--- INICIO DEL HTML RECIBIDO ---
<!DOCTYPE html><html lang="es" prefix="og: https://ogp.me/ns#"><head><script data-no-optimize="1">var litespeed_docref=sessionStorage.getItem("litespeed_docref");litespeed_docref&&(Object.defineProperty(document,"referrer",{get:function(){return litespeed_docref}}),sessionStorage.removeItem("litespeed_docref"));</script> <meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" /><link rel="profile" href="http://gmpg.org/xfn/11"><link rel="pingback" href="https://cultocervecero.com/xmlrpc.php"><style>img:is([sizes="auto" i], [sizes^="auto," i]) { contain-intrinsic-size: 3000px 1500px }</style> <script data-cfasync="false" data-pagespeed-no-defer>var gtm4wp_datalayer_name = "dataLayer";
	var dataLayer = dataLayer || [];
	const gtm4wp_use_sku_instead = 0;
	const gtm4wp_currency = 'COP';
	const gtm4wp_product_per_impression = 10;
	const gtm4wp_clear_ecommerce = false;
	const gtm4wp_data

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [ ]:

# 1. Configuración del Navegador
options = webdriver.ChromeOptions()
# options.add_argument('--headless') # Descomenta si no quieres ver la ventana
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://cultocervecero.com/categoria-producto/cervezas/cervecerias/"
lista_datos = []

try:
    print(f"Conectando a: {url}")
    driver.get(url)
    
    time.sleep(5) # Damos tiempo extra para que cargue todo
    driver.save_screenshot("error_diagnostico.png")
    print("📸 Captura de pantalla guardada como 'error_diagnostico.png'")
    wait = WebDriverWait(driver, 20)
    
    # 2. Superar la barrera de edad 🔞
    print("Esperando verificación de edad...")
    boton_si = wait.until(EC.element_to_be_clickable((By.NAME, "overAge")))
    boton_si.click()
    print("✅ Verificación de edad superada.")

    # Pausa para que el modal desaparezca por completo
    time.sleep(3) 
    
    # 3. Movimiento y carga 🖱️
    driver.execute_script("window.scrollTo(0, 1000);")
    time.sleep(2)

    # 4. Localizar los productos 🔍
    print("Buscando productos...")
    try:
        # Esperamos a que el contenedor de productos sea visible
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.product")))
        productos = driver.find_elements(By.CSS_SELECTOR, "li.product")
    except Exception:
        # Fallback por si la estructura cambia levemente
        productos = driver.find_elements(By.CLASS_NAME, "product")
    
    print(f"¡Se encontraron {len(productos)} bloques de productos!")

    # 5. Extracción de datos 🍺
    for p in productos:
        try:
            # Extraemos nombre y precio con selectores específicos de WooCommerce
            nombre = p.find_element(By.CSS_SELECTOR, ".woocommerce-loop-product__title").text.strip()
            precio = p.find_element(By.CSS_SELECTOR, ".price").text.strip()
            
            if nombre:
                lista_datos.append({
                    'Cerveza': nombre,
                    'Precio': precio.replace('\n', ' ')
                })
                print(f"✅ Extraído: {nombre}")
                
        except Exception:
            # Si falla un producto, imprimimos su HTML para debug y seguimos
            print("⚠️ No se pudo extraer info de un producto. Saltando...")
            continue

    # 6. Guardar los resultados 💾
    if lista_datos:
        df = pd.DataFrame(lista_datos)
        df.to_csv('cervezas_culto_final.csv', index=False, encoding='utf-8-sig')
        print(f"\n🚀 ¡Éxito! Se guardaron {len(df)} cervezas en 'cervezas_culto_final.csv'.")
    else:
        print("⚠️ No se pudo extraer información. Revisa si los selectores CSS cambiaron.")

except Exception as e:
    print(f"❌ Error crítico en el proceso: {e}")

finally:
    # 7. Siempre cerramos el navegador
    driver.quit()
    print("Navegador cerrado.")

Conectando a: https://cultocervecero.com/categoria-producto/cervezas/cervecerias/
📸 Captura de pantalla guardada como 'error_diagnostico.png'
Esperando verificación de edad...
✅ Verificación de edad superada.
Buscando productos...
¡Se encontraron 32 bloques de productos!
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo extraer info de un producto. Saltando...
⚠️ No se pudo